## Bronze to Silver: Data Cleaning and Transformation for Dimension Tables

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

catalog_name = 'ecommerce'

##Brands

In [0]:
df_brz_brands = spark.table(f"{catalog_name}.bronze.brz_brands")
display(df_brz_brands.show(10))

In [0]:
#trim whitespaces present in brand_name column and remove special characters from the brand_code column
df_sil_brand = df_brz_brands.withColumn("brand_name", F.trim(F.col("brand_name"))) \
    .withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))

display(df_sil_brand.show(5))

In [0]:
df_sil_brand.select("category_code").distinct().show()

In [0]:
#standardinze category_code column
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}
df_sil_brand = df_sil_brand.replace(anomalies, subset=["category_code"])

#display
df_sil_brand.select("category_code").distinct().show()


In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_sil_brand.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

##Category

In [0]:
df_brz_category = spark.table(f"{catalog_name}.bronze.brz_category")
df_brz_category.show(10)

In [0]:
#checking for duplicates
df_duplicates = df_brz_category.groupBy("category_code").count().filter(F.col("count")>1)
display(df_duplicates)

In [0]:
df_silver_category = df_brz_category.dropDuplicates(['category_code'])
display(df_silver_category)

In [0]:
df_silver_category = df_silver_category.withColumn("category_code", F.upper(F.col("category_code")))
display(df_silver_category)

In [0]:
df_silver_category.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_category")

##Products

In [0]:
 df_bronze_prdts = spark.read.table(f"{catalog_name}.bronze.brz_products")
 df_bronze_prdts.show(10)

In [0]:
# Get row and column count
row_count, column_count = df_bronze_prdts.count(), len(df_bronze_prdts.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

In [0]:
display(df_bronze_prdts)

In [0]:
columns_to_keep = [
    "product_id",
    "sku",
    "category_code",
    "brand_code",
    "color",
    "size",
    "material",
    "weight_grams",
    "length_cm",
    "width_cm",
    "height_cm",
    "rating_count",
    "_source_file",
    "ingested_at"
]

df_bronze_prdts = df_bronze_prdts.select(columns_to_keep)
display(df_bronze_prdts)

In [0]:
row_count, column_count = df_bronze_prdts.count(), len(df_bronze_prdts.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

In [0]:
from pyspark.sql.types import FloatType, IntegerType, DecimalType
from pyspark.sql import functions as F

df_silver_prdts = df_bronze_prdts.withColumn(
    "weight_grams", F.bround(F.regexp_replace("weight_grams", "g", "").cast(DecimalType(10,2)), 2)
).withColumn(
    "length_cm", F.bround(F.regexp_replace("length_cm", ",", ".").cast(DecimalType(10,2)), 2)
).withColumn(
    "width_cm", F.bround(F.regexp_replace("width_cm", ",", ".").cast(DecimalType(10,2)), 2)
).withColumn(
    "height_cm", F.bround(F.regexp_replace("height_cm", ",", ".").cast(DecimalType(10,2)), 2)
).withColumn(
    "rating_count", F.col("rating_count").cast(IntegerType())
)

display(df_silver_prdts.show(5))

In [0]:
#category and brand code are in lower case, we need them to be in upper case
df_silver_prdts = df_silver_prdts.withColumn("category_code", F.upper(F.col("category_code"))).withColumn("brand_code", F.upper(F.col("brand_code")))
display(df_silver_prdts.select("category_code", "brand_code").show(10))

In [0]:
#Check for unique values in material column
df_silver_prdts.select("material").distinct().show()

In [0]:
# we need to fix the spelling errors like "Coton"
df_silver_prdts = df_silver_prdts.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)

display(df_silver_prdts.select("material").distinct())

In [0]:
#also noticed negative values in rating column, just describe all numeric columns to check for the outliers
numeric_cols = [col for col, dtype in df_silver_prdts.dtypes if dtype.startswith(('int', 'double', 'decimal', 'float', 'bigint', 'smallint'))]
display(df_silver_prdts.select(numeric_cols).summary())

In [0]:
df_silver_prdts.filter(F.col("rating_count")<0).select("rating_count").distinct().show()

In [0]:
#convert negative rating_count to positive
df_silver_prdts = df_silver_prdts.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

In [0]:
#df_silver_prdts.filter(F.col("rating_count")<0).select("rating_count").distinct().show()

In [0]:
#write raw data to the silver layer 
df_silver_prdts.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_prdts")

##Customers

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_brz_cust = spark.read.table(f"{catalog_name}.bronze.brz_customers")

# Get row and column count
row_count, column_count = df_brz_cust.count(), len(df_brz_cust.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_brz_cust.show(10)

In [0]:
#Handle null Values in cutomers id if any
null_count = df_brz_cust.filter(F.col("customer_id").isNull()).count()
null_count

In [0]:
# There are 300 null values in customer_id column. Display some of those
df_brz_cust.filter(F.col("customer_id").isNull()).show(3)

In [0]:
# Drop rows where 'customer_id' is null
df_silver_cust = df_brz_cust.dropna(subset=["customer_id"])

# Get row count
row_count = df_silver_cust.count()
print(f"Row count after droping null values: {row_count}")

In [0]:
#Handle null values in phone
null_count = df_silver_cust.filter(F.col("phone").isNull()).count()
print(f"Number of nulls in phone: {null_count}") 

In [0]:
df_silver_cust.filter(F.col("phone").isNull()).show(3)

In [0]:
from pyspark.sql import functions as F

duplicate_count = df_silver_cust.groupBy("customer_id").count().filter(F.col("count") > 1).count()
print(f"Number of duplicate customer_id values: {duplicate_count}")

In [0]:
#So only option is to either drop the values with empty phone numbers for the time being or replace them with "Not Available" as the dataset is huge
df_silver_cust = df_silver_cust.fillna("Not Available", subset=["phone"])

# sanity check (If any nulls still exist)
df_silver_cust.filter(F.col("phone").isNull()).show()

In [0]:
df_silver_cust.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

Calendar/Date

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_brz_cal = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

# Get row and column count
row_count, column_count = df_brz_cal.count(), len(df_brz_cal.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_brz_cal.show(3)

In [0]:
df_brz_cal.printSchema()

In [0]:
#converting string to date
from pyspark.sql.functions import to_date

df_silver_cal = df_brz_cal.withColumn("date", to_date(df_brz_cal["date"],"dd-MM-yyyy"))

In [0]:
df_silver_cal.printSchema()

In [0]:
# Find duplicate rows in the DataFrame
duplicates = df_silver_cal.groupBy('date').count().filter(F.col("count") > 1)

# Show the duplicate rows
print("Total duplicated Rows: ", duplicates.count())
display(duplicates)

In [0]:
# Remove duplicate rows
df_silver_cal = df_silver_cal.dropDuplicates(['date'])

# Get row count
row_count = df_silver_cal.count()

print("Rows After removing Duplicates: ", row_count)

In [0]:
#normalize day casing
# Capitalize first letter of each word in day_name
df_silver_cal = df_silver_cal.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver_cal.show(5)

In [0]:
#convert negative week_of_year to positive
df_silver_cal = df_silver_cal.withColumn("week_of_year", F.abs(F.col("week_of_year")))  # Convert negative to positive

df_silver_cal.show(3)

In [0]:
#Enhance quarter and week_of_year column
df_silver_cal = df_silver_cal.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver_cal = df_silver_cal.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(3)

In [0]:
#rename
df_silver_cal = df_silver_cal.withColumnRenamed("week_of_year", "week")

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_calendar)
df_silver_cal.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")